# **Notebook 04 — Feature Engineering**

## Objectives

* Engineer new features from Time and Amount based on findings from Notebook 02
* Investigate and validate potential transformations
* Apply scaling appropriate for outlier-heavy data
* Apply SMOTE oversampling to training data only

## Inputs

* `outputs/v1/X_train.csv`, `outputs/v1/X_test.csv`
* `outputs/v1/y_train.csv`, `outputs/v1/y_test.csv`

## Outputs

* `outputs/v1/X_train_engineered.csv`, `outputs/v1/X_test_engineered.csv`
* `outputs/v1/X_train_resampled.csv`, `outputs/v1/y_train_resampled.csv`
* `outputs/v1/robust_scaler.pkl`
* Documentation of all transformation decisions

---
## Change working directory

In [1]:
import os

current_dir = os.getcwd()
if current_dir.endswith("notebooks"):
    os.chdir(os.path.dirname(current_dir))
print(f"Working directory: {os.getcwd()}")

Working directory: C:\Users\name\Desktop\All IN\Code Inst. Projects\credit-card-fraud-detection


In [2]:
import pandas as pd
import numpy as np
from scipy.stats import skew
import matplotlib.pyplot as plt

X_train = pd.read_csv("outputs/v1/X_train.csv")
X_test = pd.read_csv("outputs/v1/X_test.csv")
y_train = pd.read_csv("outputs/v1/y_train.csv").squeeze()
y_test = pd.read_csv("outputs/v1/y_test.csv").squeeze()

print(f"Train set: {X_train.shape}")
print(f"Test set:  {X_test.shape}")
print(f"Features:  {X_train.shape[1]}")

Train set: (226980, 30)
Test set:  (56746, 30)
Features:  30


---
## 1. Engineer New Features

Based on findings from the Data Visualisation notebook:
- **H2 validated:** Fraud rate varies by hour → create `Hour` and `Is_Night` features
- **Amount distribution:** Highly skewed → create `Amount_log` 
- **H3 validated:** V14, V12, V10 are top discriminators → create interaction features
- **PCA variance:** Aggregate statistics across PCA components may capture anomalies

In [3]:
# Time-based features
# Hour derived from Time (seconds since first transaction)
X_train['Hour'] = (X_train['Time'] / 3600) % 24
X_test['Hour'] = (X_test['Time'] / 3600) % 24

# Night flag — fraud patterns may differ at night
X_train['Is_Night'] = ((X_train['Hour'] >= 22) | (X_train['Hour'] <= 5)).astype(int)
X_test['Is_Night'] = ((X_test['Hour'] >= 22) | (X_test['Hour'] <= 5)).astype(int)

print("Time-based features created: Hour, Is_Night")
print(f"  Night transactions in train: {X_train['Is_Night'].sum():,} "
      f"({X_train['Is_Night'].mean():.2%})")

Time-based features created: Hour, Is_Night
  Night transactions in train: 37,603 (16.57%)


In [4]:
# Amount-based features
# Log transform to reduce skewness
X_train['Amount_log'] = np.log1p(X_train['Amount'])
X_test['Amount_log'] = np.log1p(X_test['Amount'])

print("Amount feature created: Amount_log")
print(f"  Amount skewness before: {skew(X_train['Amount']):.2f}")
print(f"  Amount_log skewness:    {skew(X_train['Amount_log']):.2f}")

Amount feature created: Amount_log
  Amount skewness before: 14.39
  Amount_log skewness:    0.16


In [5]:
# Interaction features — top discriminating PCA components
# V14 and V12 are the two strongest fraud signals from H3
X_train['V14_x_V12'] = X_train['V14'] * X_train['V12']
X_test['V14_x_V12'] = X_test['V14'] * X_test['V12']

# V14 and V10 — second interaction between strong signals
X_train['V14_x_V10'] = X_train['V14'] * X_train['V10']
X_test['V14_x_V10'] = X_test['V14'] * X_test['V10']

print("Interaction features created: V14_x_V12, V14_x_V10")

Interaction features created: V14_x_V12, V14_x_V10


In [6]:
# Statistical aggregate features across all PCA components
# These capture overall "shape" of each transaction in PCA space
v_cols = [f'V{i}' for i in range(1, 29)]

X_train['V_mean'] = X_train[v_cols].mean(axis=1)
X_test['V_mean'] = X_test[v_cols].mean(axis=1)

X_train['V_std'] = X_train[v_cols].std(axis=1)
X_test['V_std'] = X_test[v_cols].std(axis=1)

X_train['V_skew'] = X_train[v_cols].skew(axis=1)
X_test['V_skew'] = X_test[v_cols].skew(axis=1)

print("Aggregate features created: V_mean, V_std, V_skew")
print(f"\nTotal features now: {X_train.shape[1]} (was 30)")

Aggregate features created: V_mean, V_std, V_skew

Total features now: 38 (was 30)


### Feature Engineering Rationale

| Feature | Type | Rationale |
|---------|------|-----------|
| Hour | Time-based | Captures time-of-day fraud patterns validated in H2 |
| Is_Night | Binary flag | Night hours may have different fraud dynamics |
| Amount_log | Transform | Reduces extreme skewness of Amount distribution |
| V14_x_V12 | Interaction | Non-linear relationship between the two strongest fraud signals |
| V14_x_V10 | Interaction | Second interaction between strong discriminators |
| V_mean | Aggregate | Average PCA value per transaction — anomalies may differ |
| V_std | Aggregate | Variance across PCA components — fraud may show unusual spread |
| V_skew | Aggregate | Distribution shape — fraud transactions may have different skew patterns |

---
## 2. Investigate Transformations

Checking whether additional transformations on Amount improve its distribution for modelling.